# Data Analytics Project — Personal Cheat Sheet

Copy/paste reference for the commands and SQL patterns used in this project.

**Project:** `data_analytics`  
**Database:** `daimonupdown`  
**Shell:** Bash on Arch Linux  
**BI:** Tableau Public workflow  

Use the section that matches the task. SQL commands go inside MariaDB or inside `mariadb -e "..."`; Bash/Git commands go at the `[lab@archlabi data_analytics]$` prompt.

## 1. Git — check, stage, commit, push

In [ ]:
# Current Git status
git status

# See every tracked file
git ls-files

# See files tracked in a specific folder
git ls-files sql/
git ls-files python/

# Check GitHub remote
git remote -v

# See recent commits
git log --oneline --decorate -10

# Refresh remote information and show what is actually on origin/main
git fetch origin
git ls-tree -r --name-only origin/main | sort

# Stage, commit, push
git add <file-or-folder>
git commit -m "Describe the change"
git push


## 2. MariaDB — database basics

In [ ]:
# Show whether LOCAL INFILE is enabled on the server
sudo mariadb -e "SHOW VARIABLES LIKE 'local_infile';"

# Run SQL from the terminal
sudo mariadb -e "USE daimonupdown; SHOW TABLES;"

# Important: SELECT / SET / SHOW are SQL, not Bash.
# At the Bash prompt, use: sudo mariadb -e "...SQL..."

# Describe tables
sudo mariadb -e "USE daimonupdown; DESCRIBE employees;"

# Show database tables
sudo mariadb -e "USE daimonupdown; SHOW TABLES;"


## 3. Run a SQL file

In [ ]:
# Run a SQL script against MariaDB
sudo mariadb --local-infile=1 < sql/03_load_data.sql

# If the script explicitly needs the database name:
sudo mariadb --local-infile=1 daimonupdown < sql/03_load_data.sql


## 4. LOAD DATA LOCAL INFILE

In [ ]:
# Use --local-infile=1 on the MariaDB client.
sudo mariadb --local-infile=1 -e "
USE daimonupdown;
LOAD DATA LOCAL INFILE '/home/lab/Desktop/GitHub/data_analytics/data/raw/departments.csv'
INTO TABLE departments
FIELDS TERMINATED BY ','
ENCLOSED BY '\"'
LINES TERMINATED BY '\n'
IGNORE 1 LINES;
SELECT COUNT(*) AS department_count FROM departments;
"


## 5. Disable/enable foreign-key checks during controlled reloads

In [ ]:
# SQL must be passed to MariaDB; do NOT type SET directly at the Bash prompt.
sudo mariadb -e "
USE daimonupdown;
SET FOREIGN_KEY_CHECKS=0;
DELETE FROM clients;
SET FOREIGN_KEY_CHECKS=1;
"


## 6. Row-count validation

In [ ]:
sudo mariadb -e "
USE daimonupdown;
SELECT 'applications' AS table_name, COUNT(*) AS row_count FROM applications
UNION ALL SELECT 'assignments', COUNT(*) FROM assignments
UNION ALL SELECT 'attendance', COUNT(*) FROM attendance
UNION ALL SELECT 'candidates', COUNT(*) FROM candidates
UNION ALL SELECT 'client_feedback', COUNT(*) FROM client_feedback
UNION ALL SELECT 'clients', COUNT(*) FROM clients
UNION ALL SELECT 'compensation_history', COUNT(*) FROM compensation_history
UNION ALL SELECT 'departments', COUNT(*) FROM departments
UNION ALL SELECT 'employees', COUNT(*) FROM employees
UNION ALL SELECT 'employee_surveys', COUNT(*) FROM employee_surveys
UNION ALL SELECT 'employment_history', COUNT(*) FROM employment_history
UNION ALL SELECT 'locations', COUNT(*) FROM locations
UNION ALL SELECT 'offboarding', COUNT(*) FROM offboarding
UNION ALL SELECT 'onboarding', COUNT(*) FROM onboarding
UNION ALL SELECT 'performance', COUNT(*) FROM performance
UNION ALL SELECT 'positions', COUNT(*) FROM positions
UNION ALL SELECT 'qualifications', COUNT(*) FROM qualifications
UNION ALL SELECT 'schools', COUNT(*) FROM schools
UNION ALL SELECT 'training', COUNT(*) FROM training
UNION ALL SELECT 'visa_history', COUNT(*) FROM visa_history;
"


## 7. Check foreign-key relationships

In [ ]:
sudo mariadb -e "
SELECT
    TABLE_NAME,
    COLUMN_NAME,
    REFERENCED_TABLE_NAME,
    REFERENCED_COLUMN_NAME,
    CONSTRAINT_NAME
FROM INFORMATION_SCHEMA.KEY_COLUMN_USAGE
WHERE TABLE_SCHEMA = 'daimonupdown'
  AND REFERENCED_TABLE_NAME IS NOT NULL
ORDER BY TABLE_NAME, COLUMN_NAME;
"


## 8. Orphan-record checks

In [ ]:
sudo mariadb -e "
USE daimonupdown;

SELECT COUNT(*) AS orphaned_assignments
FROM assignments a
LEFT JOIN employees e ON a.employee_id = e.employee_id
WHERE e.employee_id IS NULL;

SELECT COUNT(*) AS orphaned_training
FROM training t
LEFT JOIN employees e ON t.employee_id = e.employee_id
WHERE e.employee_id IS NULL;

SELECT COUNT(*) AS orphaned_performance
FROM performance p
LEFT JOIN employees e ON p.employee_id = e.employee_id
WHERE e.employee_id IS NULL;

SELECT COUNT(*) AS orphaned_attendance
FROM attendance a
LEFT JOIN employees e ON a.employee_id = e.employee_id
WHERE e.employee_id IS NULL;
"


## 9. Duplicate primary-key checks

In [ ]:
sudo mariadb -e "
USE daimonupdown;
SELECT
    'employees' AS table_name,
    COUNT(*) - COUNT(DISTINCT employee_id) AS duplicate_ids
FROM employees;

# Repeat the same pattern for another table:
# SELECT COUNT(*) - COUNT(DISTINCT assignment_id) FROM assignments;
"


## 10. Find invalid or missing dates

In [ ]:
sudo mariadb -e "
USE daimonupdown;
SELECT COUNT(*) AS invalid_client_dates
FROM clients
WHERE client_relationship_end_date IS NOT NULL
  AND client_relationship_start_date > client_relationship_end_date;

SELECT COUNT(*) AS invalid_assignment_dates
FROM assignments
WHERE assignment_end_date IS NOT NULL
  AND assignment_start_date > assignment_end_date;

SELECT COUNT(*) AS invalid_employment_dates
FROM employment_history
WHERE employment_end_date IS NOT NULL
  AND employment_start_date > employment_end_date;

SELECT COUNT(*) AS invalid_onboarding_dates
FROM onboarding
WHERE onboarding_end_date IS NOT NULL
  AND onboarding_start_date > onboarding_end_date;
"


## 11. Check NULLs in important fields

In [ ]:
sudo mariadb -e "
USE daimonupdown;
SELECT 'employees.employee_id' AS field_name, COUNT(*) - COUNT(employee_id) AS null_count FROM employees
UNION ALL SELECT 'employees.candidate_id', COUNT(*) - COUNT(candidate_id) FROM employees
UNION ALL SELECT 'assignments.employee_id', COUNT(*) - COUNT(employee_id) FROM assignments
UNION ALL SELECT 'assignments.client_id', COUNT(*) - COUNT(client_id) FROM assignments
UNION ALL SELECT 'performance.employee_id', COUNT(*) - COUNT(employee_id) FROM performance
UNION ALL SELECT 'attendance.employee_id', COUNT(*) - COUNT(employee_id) FROM attendance;
"


## 12. Check category values

In [ ]:
sudo mariadb -e "
USE daimonupdown;
SELECT DISTINCT client_status FROM clients ORDER BY client_status;
SELECT DISTINCT assignment_status FROM assignments ORDER BY assignment_status;
SELECT DISTINCT employment_status FROM employment_history ORDER BY employment_status;
SELECT DISTINCT candidate_status FROM candidates ORDER BY candidate_status;
SELECT DISTINCT completion_status FROM training ORDER BY completion_status;
"


## 13. Recruitment analysis patterns

In [ ]:
sudo mariadb -e "
USE daimonupdown;
SELECT
    application_status,
    COUNT(*) AS applications
FROM applications
GROUP BY application_status
ORDER BY applications DESC;
"

# Hired applications vs distinct hired candidates
sudo mariadb -e "
USE daimonupdown;
SELECT
    COUNT(*) AS hired_applications,
    COUNT(DISTINCT candidate_id) AS hired_candidates
FROM applications
WHERE application_status = 'Hired';
"

# Application hire rate
sudo mariadb -e "
USE daimonupdown;
SELECT
    COUNT(*) AS total_applications,
    SUM(application_status = 'Hired') AS hired_applications,
    COUNT(DISTINCT CASE WHEN application_status = 'Hired' THEN candidate_id END) AS hired_candidates,
    ROUND(100.0 * SUM(application_status = 'Hired') / COUNT(*), 2) AS application_hire_rate
FROM applications;
"


## 14. Find repeat hired candidates

In [ ]:
sudo mariadb -e "
USE daimonupdown;
SELECT
    candidate_id,
    COUNT(*) AS hired_application_count
FROM applications
WHERE application_status = 'Hired'
GROUP BY candidate_id
HAVING COUNT(*) > 1
ORDER BY hired_application_count DESC;
"


## 15. Recruitment source performance

In [ ]:
sudo mariadb -e "
USE daimonupdown;
SELECT
    c.recruitment_source,
    COUNT(DISTINCT c.candidate_id) AS candidates,
    COUNT(DISTINCT CASE WHEN a.application_status = 'Hired' THEN a.candidate_id END) AS hired_candidates,
    ROUND(
        100.0 * COUNT(DISTINCT CASE WHEN a.application_status = 'Hired' THEN a.candidate_id END)
        / COUNT(DISTINCT c.candidate_id), 2
    ) AS candidate_hire_rate
FROM candidates c
LEFT JOIN applications a ON c.candidate_id = a.candidate_id
GROUP BY c.recruitment_source
ORDER BY candidate_hire_rate DESC;
"


## 16. Position performance

In [ ]:
sudo mariadb -e "
USE daimonupdown;
SELECT
    position_applied,
    COUNT(*) AS applications,
    SUM(application_status = 'Hired') AS hired_applications,
    COUNT(DISTINCT CASE WHEN application_status = 'Hired' THEN candidate_id END) AS hired_candidates,
    ROUND(100.0 * SUM(application_status = 'Hired') / COUNT(*), 2) AS application_hire_rate
FROM applications
GROUP BY position_applied
ORDER BY applications DESC;
"


## 17. Export validated MariaDB data for dashboard work

In [ ]:
# Create dashboard data folder
mkdir -p dashboard/data

# Export with headers. Do NOT use --skip-column-names.
sudo mariadb --batch -e "USE daimonupdown; SELECT * FROM employees;" > dashboard/data/employees.tsv
sudo mariadb --batch -e "USE daimonupdown; SELECT * FROM assignments;" > dashboard/data/assignments.tsv
sudo mariadb --batch -e "USE daimonupdown; SELECT * FROM clients;" > dashboard/data/clients.tsv
sudo mariadb --batch -e "USE daimonupdown; SELECT * FROM attendance;" > dashboard/data/attendance.tsv

# Convert TSV files to CSV with the reusable Python tool
python python/tsv_to_csv.py dashboard/data/

# Remove intermediate TSV files
rm dashboard/data/*.tsv


## 18. Build the Tableau-ready workforce dataset

In [ ]:
# Activate the project virtual environment
source .venv/bin/activate

# Install dependencies when needed
pip install pandas openpyxl

# Build one row per assignment
python python/create_workforce_dashboard.py


## 19. Python data-preparation tools

In [ ]:
# Convert one TSV file to CSV
python python/tsv_to_csv.py input.tsv

# Convert every TSV in a folder
python python/tsv_to_csv.py input_folder/ output_folder/

# Merge Excel workbooks in a folder.
# Worksheets with the same name are merged into one CSV.
python python/merge_excel_to_csv.py excel_folder csv_folder


## 20. Create a MariaDB backup/dump

In [ ]:
# Complete database dump
sudo mariadb-dump --databases daimonupdown > sql/daimonupdown_dump.sql

# Check dump size
ls -lh sql/daimonupdown_dump.sql

# Check beginning of dump
head -n 20 sql/daimonupdown_dump.sql


## 21. Common mistakes

**SQL at the Bash prompt:**
```text
[lab@archlabi data_analytics]$ SELECT COUNT(*) FROM departments;
bash: syntax error
```
Use `sudo mariadb -e "SELECT ..."` instead.

**Running a SQL filename as a command:**
```text
sql/03_load_data.sql
```
Use `sudo mariadb --local-infile=1 < sql/03_load_data.sql`.

**LOCAL INFILE error:**
Use `--local-infile=1` on the MariaDB client.

**TSV export without headers:**
Do not use `--skip-column-names` when the file is going to Tableau.

**Git:**
After changes: `git status` → `git add` → `git commit` → `git push` → `git status`.


## 22. Current project folders

```text
data_analytics/
├── dashboard/       # Tableau dashboard files + dashboard data
├── data/raw/        # Original CSV source data
├── docs/            # Project documentation and data model
├── images/          # Images/screenshots
├── notebooks/       # Personal cheat sheets + exploratory notebooks
├── python/          # Reusable Python data tools
├── reports/         # Reports
└── sql/             # Database creation, loading, validation, analysis, dump
```
